In [ ]:
from kafka.admin import KafkaAdminClient, NewTopic

In [ ]:
def create_kafka_topic(topic_name):
    admin_client = KafkaAdminClient(
        bootstrap_servers="localhost:9092",
        client_id='kafka-python-topic-creator'
    )
    topic_list = [NewTopic(name=topic_name, num_partitions=1, replication_factor=1)]
    try:
        admin_client.create_topics(new_topics=topic_list, validate_only=False)
        print(f"Topic '{topic_name}' created successfully.")
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        admin_client.close()


topic_name = 'video-stream'
create_kafka_topic(topic_name)

In [ ]:
from kafka import KafkaProducer
import cv2
import numpy as np
import time
import logging

In [ ]:
producer = KafkaProducer(bootstrap_servers='localhost:9092')

In [ ]:
def publish_video():
    
    producer = KafkaProducer(bootstrap_servers='localhost:9092')
    video_capture = cv2.VideoCapture(0)
    try:
        logging.info("Producer started...")
        frame_count = 0
        while True:
            ret, frame = video_capture.read()
            if not ret:
                logging.error("Failed to capture frame.")
                break
            
            ret, buffer = cv2.imencode('.jpg', frame)
            if not ret:
                logging.error("Failed to encode frame.")
                break
            
            future = producer.send(topic_name, buffer.tobytes())
            
            frame_count += 1
            logging.info(f"Sent frame {frame_count}")
            
            time.sleep(0.1)
    except Exception as e:
        logging.error(f"An error occurred in the producer: {e}")
    finally:
        
        producer.flush()
        producer.close()
        video_capture.release()
        logging.info("Producer closed.")


In [ ]:
publish_video()